# 🚀 NEGATIVE BINOMIAL ULTIMATE - L'ARME FINALE
## Objectif : Exploser le 0.77 avec NegBin + Smart Features + Mix

### Stratégie :
1. **Negative Binomial Pur** (count:poisson) avec Smart Features
2. **Negative Binomial + LogLoss Mix** (Hybride comme Ultimate Mix)
3. **Comparaison exhaustive** avec toutes les métriques

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, 
    recall_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
import xgboost as xgb

print("⏳ Chargement des données...")
train_data = pd.read_csv('conversion_data_train.csv')
test_data = pd.read_csv('conversion_data_test.csv')

# Smart Feature Engineering (Version ULTIMATE)
def smart_feature_engineering(df):
    df_eng = df.copy()
    
    # Features Critiques
    df_eng['is_active'] = (df_eng['total_pages_visited'] > 2).astype(int)
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    
    return df_eng

X = smart_feature_engineering(train_data.drop('converted', axis=1))
y = train_data['converted']
X_test = smart_feature_engineering(test_data)

# Encodage
categorical_cols = ['country', 'source']
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]])
    le.fit(combined)
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

print(f"✅ Dataset prêt : {len(X)} lignes, {X.shape[1]} features")
print(f"   Features : {list(X.columns)}")
print(f"   Taux de conversion : {y.mean():.4f}\n")

⏳ Chargement des données...
✅ Dataset prêt : 284580 lignes, 8 features
   Features : ['country', 'age', 'new_user', 'source', 'total_pages_visited', 'is_active', 'interaction_age_pages', 'pages_per_age']
   Taux de conversion : 0.0323



In [2]:
# ====================================================================
# MODÈLE 1 : NEGATIVE BINOMIAL PUR (avec Smart Features)
# ====================================================================
print("🧪 MODÈLE 1 : NEGATIVE BINOMIAL PUR + SMART FEATURES")
print("="*70)

n_folds = 10
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_preds_negbin = np.zeros(len(X))
test_preds_negbin = np.zeros(len(X_test))
fold_scores_negbin = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    # Negative Binomial (count:poisson avec max_delta_step)
    params_nb = {
        'objective': 'count:poisson',
        'max_delta_step': 0.7,
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42 + fold
    }
    
    dtrain = xgb.DMatrix(X_train_fold, label=y_train_fold)
    dval = xgb.DMatrix(X_val_fold)
    dtest = xgb.DMatrix(X_test)
    
    model_nb = xgb.train(params_nb, dtrain, num_boost_round=500, verbose_eval=False)
    
    oof_preds_negbin[val_idx] = model_nb.predict(dval)
    test_preds_negbin += model_nb.predict(dtest)
    
    auc = roc_auc_score(y_val_fold, oof_preds_negbin[val_idx])
    fold_scores_negbin.append(auc)
    print(f"  > Fold {fold+1}/{n_folds} | ROC-AUC: {auc:.5f}")

test_preds_negbin /= n_folds

# Optimisation seuil
best_f1_nb = 0
best_thresh_nb = 0
for t in np.linspace(oof_preds_negbin.min(), oof_preds_negbin.max(), 1000):
    preds = (oof_preds_negbin >= t).astype(int)
    score = f1_score(y, preds)
    if score > best_f1_nb:
        best_f1_nb = score
        best_thresh_nb = t

# Métriques finales
final_preds_nb = (oof_preds_negbin >= best_thresh_nb).astype(int)
metrics_nb = {
    'f1': best_f1_nb,
    'roc_auc': np.mean(fold_scores_negbin),
    'precision': precision_score(y, final_preds_nb),
    'recall': recall_score(y, final_preds_nb),
    'accuracy': accuracy_score(y, final_preds_nb),
    'threshold': best_thresh_nb
}

print(f"\n📊 RÉSULTATS NEGATIVE BINOMIAL PUR :")
print(f"   F1-Score   : {metrics_nb['f1']:.5f}")
print(f"   ROC-AUC    : {metrics_nb['roc_auc']:.5f}")
print(f"   Precision  : {metrics_nb['precision']:.5f}")
print(f"   Recall     : {metrics_nb['recall']:.5f}")
print(f"   Accuracy   : {metrics_nb['accuracy']:.5f}")
print(f"   Seuil      : {metrics_nb['threshold']:.6f}\n")

🧪 MODÈLE 1 : NEGATIVE BINOMIAL PUR + SMART FEATURES
  > Fold 1/10 | ROC-AUC: 0.98635
  > Fold 2/10 | ROC-AUC: 0.98378
  > Fold 3/10 | ROC-AUC: 0.98340
  > Fold 4/10 | ROC-AUC: 0.98660
  > Fold 5/10 | ROC-AUC: 0.98741
  > Fold 6/10 | ROC-AUC: 0.98423
  > Fold 7/10 | ROC-AUC: 0.98275
  > Fold 8/10 | ROC-AUC: 0.98454
  > Fold 9/10 | ROC-AUC: 0.98750
  > Fold 10/10 | ROC-AUC: 0.98582

📊 RÉSULTATS NEGATIVE BINOMIAL PUR :
   F1-Score   : 0.76788
   ROC-AUC    : 0.98524
   Precision  : 0.81678
   Recall     : 0.72451
   Accuracy   : 0.98587
   Seuil      : 0.409436



In [3]:
# ====================================================================
# MODÈLE 2 : NEGATIVE BINOMIAL + LOGLOSS MIX (Hybride)
# ====================================================================
print("🧪 MODÈLE 2 : NEGATIVE BINOMIAL + LOGLOSS MIX")
print("="*70)

oof_preds_mix = np.zeros(len(X))
test_preds_mix = np.zeros(len(X_test))
fold_scores_mix = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    # MODÈLE A : Negative Binomial
    params_nb = {
        'objective': 'count:poisson',
        'max_delta_step': 0.7,
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42 + fold
    }
    
    dtrain = xgb.DMatrix(X_train_fold, label=y_train_fold)
    dval = xgb.DMatrix(X_val_fold)
    dtest = xgb.DMatrix(X_test)
    
    model_nb = xgb.train(params_nb, dtrain, num_boost_round=500, verbose_eval=False)
    val_pred_nb = model_nb.predict(dval)
    test_pred_nb = model_nb.predict(dtest)
    
    # MODÈLE B : LogLoss Classifier
    cat_indices = [X.columns.get_loc(col) for col in categorical_cols]
    model_clf = HistGradientBoostingClassifier(
        loss='log_loss', learning_rate=0.05, max_iter=500, max_depth=8,
        l2_regularization=0.1, categorical_features=cat_indices, random_state=42+fold
    )
    model_clf.fit(X_train_fold, y_train_fold)
    val_pred_clf = model_clf.predict_proba(X_val_fold)[:, 1]
    test_pred_clf = model_clf.predict_proba(X_test)[:, 1]
    
    # MIX
    val_mix = (val_pred_nb + val_pred_clf) / 2
    test_mix = (test_pred_nb + test_pred_clf) / 2
    
    oof_preds_mix[val_idx] = val_mix
    test_preds_mix += test_mix
    
    auc = roc_auc_score(y_val_fold, val_mix)
    fold_scores_mix.append(auc)
    print(f"  > Fold {fold+1}/{n_folds} | Mix ROC-AUC: {auc:.5f}")

test_preds_mix /= n_folds

# Optimisation seuil
best_f1_mix = 0
best_thresh_mix = 0
for t in np.linspace(oof_preds_mix.min(), oof_preds_mix.max(), 1000):
    preds = (oof_preds_mix >= t).astype(int)
    score = f1_score(y, preds)
    if score > best_f1_mix:
        best_f1_mix = score
        best_thresh_mix = t

# Métriques finales
final_preds_mix = (oof_preds_mix >= best_thresh_mix).astype(int)
metrics_mix = {
    'f1': best_f1_mix,
    'roc_auc': np.mean(fold_scores_mix),
    'precision': precision_score(y, final_preds_mix),
    'recall': recall_score(y, final_preds_mix),
    'accuracy': accuracy_score(y, final_preds_mix),
    'threshold': best_thresh_mix
}

print(f"\n📊 RÉSULTATS NEGBIN + LOGLOSS MIX :")
print(f"   F1-Score   : {metrics_mix['f1']:.5f}")
print(f"   ROC-AUC    : {metrics_mix['roc_auc']:.5f}")
print(f"   Precision  : {metrics_mix['precision']:.5f}")
print(f"   Recall     : {metrics_mix['recall']:.5f}")
print(f"   Accuracy   : {metrics_mix['accuracy']:.5f}")
print(f"   Seuil      : {metrics_mix['threshold']:.6f}\n")

🧪 MODÈLE 2 : NEGATIVE BINOMIAL + LOGLOSS MIX
  > Fold 1/10 | Mix ROC-AUC: 0.98668
  > Fold 2/10 | Mix ROC-AUC: 0.98399
  > Fold 3/10 | Mix ROC-AUC: 0.98372
  > Fold 4/10 | Mix ROC-AUC: 0.98667
  > Fold 5/10 | Mix ROC-AUC: 0.98773
  > Fold 6/10 | Mix ROC-AUC: 0.98444
  > Fold 7/10 | Mix ROC-AUC: 0.98297
  > Fold 8/10 | Mix ROC-AUC: 0.98484
  > Fold 9/10 | Mix ROC-AUC: 0.98786
  > Fold 10/10 | Mix ROC-AUC: 0.98620

📊 RÉSULTATS NEGBIN + LOGLOSS MIX :
   F1-Score   : 0.76868
   ROC-AUC    : 0.98551
   Precision  : 0.83045
   Recall     : 0.71547
   Accuracy   : 0.98611
   Seuil      : 0.439069



In [4]:
# ====================================================================
# TABLEAU COMPARATIF EXHAUSTIF
# ====================================================================
print("\n" + "="*90)
print("🏆 COMPARAISON EXHAUSTIVE - TOUS LES MODÈLES")
print("="*90)

# Références (scores précédents)
results = [
    {
        'Modèle': 'Ultimate Mix (Poisson+LogLoss)',
        'F1': 0.76877,
        'ROC-AUC': 0.98544,
        'Precision': 0.82036,
        'Recall': 0.72081,
        'Accuracy': 0.96784,
        'Statut': 'REF'
    },
    {
        'Modèle': 'NegBin Pur + Smart Features',
        'F1': metrics_nb['f1'],
        'ROC-AUC': metrics_nb['roc_auc'],
        'Precision': metrics_nb['precision'],
        'Recall': metrics_nb['recall'],
        'Accuracy': metrics_nb['accuracy'],
        'Statut': 'NEW'
    },
    {
        'Modèle': 'NegBin + LogLoss Mix',
        'F1': metrics_mix['f1'],
        'ROC-AUC': metrics_mix['roc_auc'],
        'Precision': metrics_mix['precision'],
        'Recall': metrics_mix['recall'],
        'Accuracy': metrics_mix['accuracy'],
        'Statut': 'NEW'
    },
    {
        'Modèle': 'Mariage Frère (Tweedie)',
        'F1': 0.76666,
        'ROC-AUC': 0.98531,
        'Precision': 0.79893,
        'Recall': 0.73235,
        'Accuracy': 0.96750,
        'Statut': 'REF'
    }
]

# Tri par F1-Score
results_sorted = sorted(results, key=lambda x: x['F1'], reverse=True)

print(f"\n{'Rang':<6} {'Modèle':<35} {'F1':<10} {'ROC-AUC':<10} {'Prec':<10} {'Recall':<10} {'Acc':<10}")
print("-"*90)

for i, r in enumerate(results_sorted, 1):
    emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    status_emoji = "🆕" if r['Statut'] == 'NEW' else "📌"
    
    print(f"{emoji} {i:<4} {r['Modèle']:<35} {r['F1']:.5f}  {r['ROC-AUC']:.5f}  "
          f"{r['Precision']:.5f}  {r['Recall']:.5f}  {r['Accuracy']:.5f}  {status_emoji}")

print("\n" + "="*90)

# Analyse du vainqueur
winner = results_sorted[0]
if winner['Statut'] == 'NEW' and winner['F1'] > 0.76877:
    gain = (winner['F1'] - 0.76877) * 100
    print(f"🎉 VICTOIRE ! '{winner['Modèle']}' bat Ultimate Mix !")
    print(f"   Gain F1-Score : +{gain:.3f} points")
    print(f"   💡 Recommandation : SOUMETTRE CE MODÈLE IMMÉDIATEMENT !")
    
    # Génération automatique
    if 'NegBin Pur' in winner['Modèle']:
        final_test = (test_preds_negbin >= best_thresh_nb).astype(int)
        filename = 'submission_NEGBIN_ULTIMATE.csv'
    else:
        final_test = (test_preds_mix >= best_thresh_mix).astype(int)
        filename = 'submission_NEGBIN_MIX.csv'
    
    submission = pd.DataFrame({'converted': final_test})
    submission.to_csv(filename, index=False)
    print(f"   ✅ Fichier '{filename}' généré avec {final_test.sum()} conversions.")
    
elif winner['F1'] == 0.76877:
    print(f"📊 Égalité parfaite avec Ultimate Mix.")
    print(f"   Les deux approches sont équivalentes.")
else:
    print(f"📊 Ultimate Mix reste champion.")
    print(f"   Mais NegBin est très proche ! Écart : {(0.76877 - winner['F1'])*100:.3f} points")

print("="*90)


🏆 COMPARAISON EXHAUSTIVE - TOUS LES MODÈLES

Rang   Modèle                              F1         ROC-AUC    Prec       Recall     Acc       
------------------------------------------------------------------------------------------
🥇 1    Ultimate Mix (Poisson+LogLoss)      0.76877  0.98544  0.82036  0.72081  0.96784  📌
🥈 2    NegBin + LogLoss Mix                0.76868  0.98551  0.83045  0.71547  0.98611  🆕
🥉 3    NegBin Pur + Smart Features         0.76788  0.98524  0.81678  0.72451  0.98587  🆕
   4    Mariage Frère (Tweedie)             0.76666  0.98531  0.79893  0.73235  0.96750  📌

📊 Égalité parfaite avec Ultimate Mix.
   Les deux approches sont équivalentes.
